# City Pulse — Extension Challenges
### Advanced techniques for students who've completed the main walkthrough

**Prerequisites:** Complete `01_city_pulse_walkthrough.ipynb` first.

**Topics:** Random Forest · SHAP explainability · DBSCAN · Hyperparameter tuning · City recommender · Live API data

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from src.data.loader import load_raw, clean, add_features, NUMERIC_FEATURES, TARGET, FEATURE_DISPLAY_NAMES

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

df = add_features(clean(load_raw()))
scaler = StandardScaler()
X = df[NUMERIC_FEATURES]
y = df[TARGET]
X_train_r, X_test_r, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = pd.DataFrame(scaler.fit_transform(X_train_r), columns=NUMERIC_FEATURES, index=X_train_r.index)
X_test  = pd.DataFrame(scaler.transform(X_test_r),     columns=NUMERIC_FEATURES, index=X_test_r.index)
print('Ready. Train:', len(X_train), '| Test:', len(X_test))

---
## Challenge 1 — Random Forest vs Linear Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        'R²': round(r2_score(y_test, pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 4),
    }

pd.DataFrame(results).T

In [ ]:
# Feature importances — Random Forest vs Lasso
from sklearn.linear_model import Lasso

rf = models['Random Forest']
lasso = Lasso(alpha=0.1).fit(X_train, y_train)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
feat_names = [FEATURE_DISPLAY_NAMES[f] for f in NUMERIC_FEATURES]

# RF importances
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values()
ax1.barh(imp.index, imp.values, color='#7F77DD')
ax1.set_title('Random Forest — Feature Importances', fontweight='bold')
ax1.set_xlabel('Importance')

# Lasso coefficients
coef = pd.Series(lasso.coef_, index=feat_names).sort_values()
colors = ['#D85A30' if v < 0 else '#1D9E75' for v in coef.values]
ax2.barh(coef.index, coef.values, color=colors)
ax2.axvline(0, color='#888', lw=0.8)
ax2.set_title('Lasso (α=0.1) — Coefficients', fontweight='bold')
ax2.set_xlabel('Coefficient')

fig.suptitle('Which features drive liveability?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Discussion:** Do the two methods agree on which features matter most? Why might they disagree? Which is more trustworthy for a small dataset?

---
## Challenge 2 — SHAP Explainability

In [ ]:
# pip install shap
try:
    import shap
    shap.initjs()

    rf = models['Random Forest']
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)

    print('=== Global feature importance (SHAP) ===')
    shap.summary_plot(
        shap_values, X_test,
        feature_names=[FEATURE_DISPLAY_NAMES[f] for f in NUMERIC_FEATURES],
        plot_type='bar',
    )

    print('\n=== Beeswarm: direction and magnitude ===')
    shap.summary_plot(
        shap_values, X_test,
        feature_names=[FEATURE_DISPLAY_NAMES[f] for f in NUMERIC_FEATURES],
    )
except ImportError:
    print('Install shap first:  pip install shap')

In [ ]:
# Explain a single city's prediction
try:
    city_name = 'Tokyo'
    idx = df[df['city'] == city_name].index[0]
    city_X = X.loc[[idx]]
    city_X_scaled = pd.DataFrame(scaler.transform(city_X), columns=NUMERIC_FEATURES)

    shap_single = explainer.shap_values(city_X_scaled)
    print(f'Predicted liveability for {city_name}: {rf.predict(city_X_scaled)[0]:.1f}')
    print(f'Actual liveability: {df.loc[idx, TARGET]}')

    shap.force_plot(
        explainer.expected_value,
        shap_single[0],
        feature_names=[FEATURE_DISPLAY_NAMES[f] for f in NUMERIC_FEATURES],
        matplotlib=True,
    )
except Exception as e:
    print('Run the SHAP install cell above first.', e)

---
## Challenge 3 — Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Build a pipeline so scaling happens inside CV (no leakage)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge()),
])

param_grid = {'ridge__alpha': [0.01, 0.1, 1, 5, 10, 50, 100, 500]}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', return_train_score=True)
grid.fit(X, y)  # use unscaled X — pipeline handles it

print(f'Best alpha: {grid.best_params_["ridge__alpha"]}')
print(f'Best CV R²: {grid.best_score_:.4f}')

# Plot CV results
results_df = pd.DataFrame(grid.cv_results_)
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(param_grid['ridge__alpha'], results_df['mean_train_score'], 'o-', label='Train R²', color='#7F77DD')
ax.semilogx(param_grid['ridge__alpha'], results_df['mean_test_score'], 's-', label='Val R²', color='#EF9F27')
ax.fill_between(param_grid['ridge__alpha'],
                results_df['mean_test_score'] - results_df['std_test_score'],
                results_df['mean_test_score'] + results_df['std_test_score'],
                alpha=0.2, color='#EF9F27')
ax.axvline(grid.best_params_['ridge__alpha'], color='#D85A30', ls='--', lw=1.5, label='Best α')
ax.set_xlabel('Alpha'); ax.set_ylabel('R²'); ax.legend()
ax.set_title('GridSearchCV — Ridge Alpha Tuning', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Challenge 4 — DBSCAN: Density-Based Clustering

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

X_all = StandardScaler().fit_transform(df[NUMERIC_FEATURES])

# Try different eps values
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pca_coords = PCA(n_components=2, random_state=42).fit_transform(X_all)

for ax, eps in zip(axes, [0.8, 1.2, 1.8]):
    dbs = DBSCAN(eps=eps, min_samples=3)
    labels = dbs.fit_predict(X_all)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()

    colors = ['#888' if l == -1 else ['#7F77DD','#1D9E75','#EF9F27','#D85A30','#378ADD'][l % 5] for l in labels]
    ax.scatter(pca_coords[:, 0], pca_coords[:, 1], c=colors, s=70, alpha=0.85, edgecolors='white', lw=0.5)

    for i, row in df.iterrows():
        if labels[df.index.get_loc(i)] == -1:
            ax.annotate(row['city'], pca_coords[df.index.get_loc(i)], fontsize=7, color='#444')

    ax.set_title(f'DBSCAN eps={eps}\n{n_clusters} clusters · {n_noise} noise pts', fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

fig.suptitle('DBSCAN — Effect of eps on cluster formation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> **Discussion:** Which cities are flagged as noise points (grey)? Does that match your intuition? How does DBSCAN's approach differ fundamentally from K-Means?

---
## Challenge 5 — City Recommender System

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_cities(
    preferences: dict,
    df: pd.DataFrame,
    top_n: int = 5,
    exclude: list = None,
) -> pd.DataFrame:
    """
    Given a user's ideal feature scores, find the most similar cities.

    preferences: dict like {'air_quality': 90, 'green_space': 80, 'cost_of_living': 40}
    Missing features default to the dataset mean.
    """
    exclude = exclude or []
    # Build ideal vector (fill missing with mean)
    ideal = {f: preferences.get(f, df[f].mean()) for f in NUMERIC_FEATURES}
    ideal_vec = np.array([ideal[f] for f in NUMERIC_FEATURES]).reshape(1, -1)

    # Normalise everything
    scaler = StandardScaler()
    city_vecs = scaler.fit_transform(df[NUMERIC_FEATURES])
    ideal_scaled = scaler.transform(ideal_vec)

    sims = cosine_similarity(ideal_scaled, city_vecs)[0]
    df_copy = df.copy()
    df_copy['similarity'] = sims
    df_copy = df_copy[~df_copy['city'].isin(exclude)]
    top = df_copy.nlargest(top_n, 'similarity')[['city', 'country', 'continent'] + NUMERIC_FEATURES + [TARGET, 'similarity']]
    top['similarity'] = top['similarity'].round(4)
    return top.reset_index(drop=True)


# Example: someone who loves clean air, green space, and safety but is on a budget
prefs = {
    'air_quality': 90,
    'green_space': 85,
    'safety_index': 90,
    'cost_of_living': 35,   # wants it cheap
    'noise_level': 20,      # wants quiet
}

print('Your ideal city profile:', {FEATURE_DISPLAY_NAMES[k]: v for k, v in prefs.items()})
print()
recommend_cities(prefs, df)

In [ ]:
# Interactive version — try your own preferences!
my_preferences = {
    'air_quality': 75,        # 0=terrible, 100=pristine
    'green_space': 60,        # 0=concrete jungle, 100=forest city
    'transit_score': 85,      # 0=no public transport, 100=world-class
    'safety_index': 70,       # 0=dangerous, 100=very safe
    'cost_of_living': 55,     # 0=incredibly cheap, 100=extremely expensive
    'noise_level': 40,        # 0=silent, 100=deafening
}

recommend_cities(my_preferences, df, top_n=8)

---
## Challenge 6 — Fetch Real Air Quality Data (OpenAQ API)

In [ ]:
import urllib.request, json

def fetch_openaq_pm25(city: str, limit: int = 10) -> float | None:
    """
    Fetch the latest PM2.5 reading for a city from the OpenAQ API.
    Returns a normalised 0-100 air quality score (higher = cleaner).
    """
    city_encoded = city.replace(' ', '%20')
    url = f'https://api.openaq.org/v2/measurements?city={city_encoded}&parameter=pm25&limit={limit}&sort=desc'
    try:
        with urllib.request.urlopen(url, timeout=5) as resp:
            data = json.loads(resp.read())
        results = data.get('results', [])
        if not results:
            return None
        avg_pm25 = np.mean([r['value'] for r in results if r['value'] > 0])
        # WHO guideline: 5 µg/m³ = very clean, 75+ = very polluted
        score = max(0, min(100, 100 - (avg_pm25 - 5) * (95 / 70)))
        return round(score, 1)
    except Exception as e:
        print(f'  Error fetching {city}: {e}')
        return None


# Try a few cities
test_cities = ['Copenhagen', 'Delhi', 'Tokyo', 'New York', 'Lagos']
print('Fetching real PM2.5 air quality data from OpenAQ...')
real_scores = {}
for city in test_cities:
    score = fetch_openaq_pm25(city)
    simulated = df.loc[df['city'] == city, 'air_quality'].values
    sim_val = simulated[0] if len(simulated) else 'N/A'
    real_scores[city] = {'Real (OpenAQ)': score, 'Simulated': sim_val}
    print(f'  {city:15s}: real={score}, simulated={sim_val}')

print("\nNote: discrepancies reflect that our dataset is educational, not live.")